# Day 2-01｜先用寫死的 Detection Box 體驗 BBOX-to-BEV

> Python「黑客松」實戰分析籃球運動數據  
> Notebook 以初學 Python 學員為主：先觀察、再改變少量變數、最後才串接。

目標：還不跑模型，先用一個寫死的 player box，理解 bottom-center 為什麼可以代表球員在地板上的位置。

In [ ]:
from pathlib import Path
import sys

# 在 Colab 中先掛載 Drive；本機執行時會自動略過。
try:
    from google.colab import drive

    drive.mount("/content/drive")
except Exception:
    pass

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機或 zip 解壓後執行時，從目前資料夾往上找。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COURSE_ROOT =", COURSE_ROOT)
print("assets =", COURSE_ROOT / "assets")

In [ ]:
from src.cv_utils import (
    read_image_rgb,
    draw_boxes,
    draw_points,
    show_image,
    side_by_side,
    load_json,
    bottom_center,
    save_image_rgb,
)
from src.geometry_utils import compute_homography, project_points

image = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png")
bev = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_bev_court.png")
info = load_json(COURSE_ROOT / "assets" / "samples" / "sample_homography_points.json")
H = compute_homography(info["camera_points"], info["bev_points"])

bbox = info["sample_player_bbox_xyxy"]
foot = bottom_center(bbox)
bev_foot = project_points([foot], H)[0]

print("bbox:", bbox)
print("bottom-center:", foot)
print("projected BEV:", bev_foot)

In [ ]:
frame_vis = draw_boxes(image, [bbox], ["player bbox"])
frame_vis = draw_points(frame_vis, [foot], ["bottom-center"])
bev_vis = draw_points(bev, [bev_foot], ["player"])
out = side_by_side(frame_vis, bev_vis)
show_image(out, "左：一個人框；右：投影到 BEV 的點")

save_path = COURSE_ROOT / "assets" / "results" / "d2_01_manual_box_to_bev.png"
save_image_rgb(save_path, out)
print("saved:", save_path)

小練習：修改 `bbox` 四個數字，觀察 BEV 點的位置怎麼變。